# 🛡️ BƯỚC 3 — Baseline 2 (B2): Phòng Thủ DP-SGD (Differential Privacy) trên Client

### 🎯 Mục Tiêu & Ý Nghĩa Khoa Học Cốt Lõi:
- **Định vị trong Luận văn**: Differential Privacy (DP-SGD, Abadi et al., 2016) là chuẩn mực bảo mật toán học khắt khe nhất trong Deep Learning.
  - Tại Client: Thực hiện **Gradient Clipping** với ngưỡng $C$ và **Cộng nhiễu Gauss** $\sigma_{\mathrm{DP}} \cdot C$ trước khi cập nhật trọng số.
  - Đo lường ngân sách bảo mật $(\epsilon, \delta)$ tích lũy qua Rényi Differential Privacy (RDP) accountant.
- **Luận điểm phản đề then chốt (Core Thesis Finding)**:
  - DP-SGD bảo đảm tính riêng tư cho bộ dữ liệu đối với *trọng số mô hình*, nhưng **HOÀN TOÀN THẤT BẠI trong việc bảo vệ biểu diễn trung gian $z = F_c(x)$** truyền qua mạng trong Split Learning.
  - Khi tăng $\sigma_{\mathrm{DP}}$ từ $0.5 \to 2.0$ (tương ứng $\epsilon$ giảm từ $\approx 25 \to 1.33$):
    1. **Utility bị phá hủy nghiêm trọng**: Độ chính xác phân loại giảm mạnh.
    2. **Security không được đảm bảo**: Kẻ tấn công (Server tò mò) huấn luyện Decoder thích ứng trên $z = F_c(x)$ vẫn tái tạo lại ảnh với chất lượng trực quan cao (PSNR $\approx 20$ dB, SSIM $> 0.65$).
  - **Kết luận**: DP-SGD không thể thay thế cơ chế bảo vệ biểu diễn trung gian $\implies$ Chứng minh tính cấp thiết và vượt trội của **Task-Aware Perceptual Encryption (Bước 5)**.

---  
## 1. Kiểm tra Môi trường & GPU (Tesla T4 / V100 / A100)

In [ ]:
!nvidia-smi

import torch
print(f"\nPyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Thiết bị GPU    : {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ CẢNH BÁO: Hãy bật GPU trong 'Runtime' -> 'Change runtime type' -> 'T4 GPU'!")

---  
## 2. Kết nối Google Drive (Lưu Checkpoints & Bảng Kết Quả B2)

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
DRIVE_B2_DIR = '/content/drive/MyDrive/AbReTAPE_Step3_B2'
os.makedirs(DRIVE_B2_DIR, exist_ok=True)
print(f"✅ Thư mục lưu kết quả Baseline B2 trên Google Drive: {DRIVE_B2_DIR}")

---  
## 3. Cài đặt Thư viện Phụ trợ & Kiểm tra Mã Nguồn

In [ ]:
!pip install -q -r requirements.txt

import os
if not os.path.exists('src'):
    !git clone https://github.com/CuongBien/AbReTAPE.git /content/AbReTAPE
    %cd /content/AbReTAPE
else:
    print("✅ Thư mục mã nguồn đã sẵn sàng!")

!ls -la src


---  
## 4. Tải CIFAR-10 & Chạy Kiểm Thử Đơn Vị (Smoke Test Baseline B2)

In [ ]:
# Chạy smoke test toàn bộ module B2 (RDP, Clipping, Noise, SL DP-SGD, Attack)
!python run_tests.py


---  
## 5. Huấn luyện Đơn Lẻ 1 Mức Nhiễu (Ví dụ: $\sigma_{\mathrm{DP}} = 1.0, C = 1.0$)
> **Gợi ý**: Chạy cell này nếu bạn muốn chạy thử nghiệm nhanh một mức $\sigma_{\mathrm{DP}}$ cụ thể.

In [ ]:
!python run_step3_baselines.py --defense b2 \
    --sigma-dp 1.0 \
    --clip-norm 1.0 \
    --micro-batch-size 16 \
    --epochs 100 \
    --decoder-epochs 30 \
    --batch-size 128 \
    --lr 0.05 \
    --decoder-lr 0.001 \
    --eval-freq 5 \
    --data-dir data \
    --output-dir /content/drive/MyDrive/AbReTAPE_Step3_B2


---  
## 6. Chạy Quét Toàn Diện Đa Mức Nhiễu DP-SGD (Sweep $\sigma_{\mathrm{DP}} \in \{0.5, 1.0, 2.0\}$)
- Tự động tính toán ngân sách bảo mật $(\epsilon, \delta)$ tương ứng.
- Huấn luyện SL với Client DP-SGD (100 epochs) và Decoder thích ứng (30 epochs) cho mỗi mức.
- Xuất bảng số liệu tổng hợp `results_b2.csv`, đồ thị `b2_tradeoff_curves.png` và lưới so sánh `b2_reconstruction_comparison.png`.

In [ ]:
!python run_step3_baselines.py --defense b2 \
    --sweep \
    --sigmas-dp 0.5,1.0,2.0 \
    --clip-norm 1.0 \
    --micro-batch-size 16 \
    --epochs 100 \
    --decoder-epochs 30 \
    --batch-size 128 \
    --lr 0.05 \
    --decoder-lr 0.001 \
    --eval-freq 5 \
    --data-dir data \
    --output-dir /content/drive/MyDrive/AbReTAPE_Step3_B2


---  
## 7. Trực Quan Hóa Đồ Thị Privacy-Utility Trade-off (DP-SGD B2)

In [ ]:
from IPython.display import Image, display
import os

tradeoff_file = "/content/drive/MyDrive/AbReTAPE_Step3_B2/b2_tradeoff_curves.png"
if not os.path.exists(tradeoff_file):
    tradeoff_file = "checkpoints_b2/b2_tradeoff_curves.png"

if os.path.exists(tradeoff_file):
    print("📈 ĐỒ THỊ PRIVACY - UTILITY TRADE-OFF (ACCURACY & PSNR/SSIM THEO SIGMA_DP VÀ EPSILON):")
    display(Image(filename=tradeoff_file, width=950))
else:
    print(f"⚠️ Chưa tìm thấy file đồ thị tại: {tradeoff_file}")

---  
## 8. Trực Quan Hóa Lưới Ảnh So Sánh Tái Tạo Trực Quan

In [ ]:
recons_file = "/content/drive/MyDrive/AbReTAPE_Step3_B2/b2_reconstruction_comparison.png"
if not os.path.exists(recons_file):
    recons_file = "checkpoints_b2/b2_reconstruction_comparison.png"

if os.path.exists(recons_file):
    print("🖼️ LƯỚI ẢNH SO SÁNH CHẤT LƯỢNG TÁI TẠO (GỐC vs CÁC MỨC DP-SGD):")
    display(Image(filename=recons_file, width=950))
else:
    print(f"⚠️ Chưa tìm thấy file lưới ảnh tại: {recons_file}")

---  
## 9. Hiển Thị Bảng Tổng Hợp Kết Quả B2 (Pandas DataFrame)

In [ ]:
import pandas as pd
import json
import os

csv_path = "/content/drive/MyDrive/AbReTAPE_Step3_B2/results_b2.csv"
if not os.path.exists(csv_path):
    csv_path = "output/AbReTAPE_Step3_B2/results_b2.csv"
if not os.path.exists(csv_path):
    csv_path = "output/AbReTAPE_Step3_B2/results_b2.json"

if os.path.exists(csv_path):
    if csv_path.endswith('.json'):
        with open(csv_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        df = pd.DataFrame(data)
    else:
        df = pd.read_csv(csv_path)
    df['test_acc'] = (df['test_acc'] * 100).round(2).astype(str) + '%'
    df['epsilon'] = df['epsilon'].round(2)
    df['psnr'] = df['psnr'].round(2).astype(str) + ' dB'
    df['ssim'] = df['ssim'].round(4)
    if 'lpips' in df.columns and df['lpips'].notnull().any():
        df['lpips'] = df['lpips'].round(4)
    print("📊 BẢNG TỔNG HỢP KẾT QUẢ BASELINE B2 (DP-SGD ON CLIENT):")
    display(df)
else:
    print(f"⚠️ Chưa tìm thấy file kết quả tại: {csv_path}")


---  
## 10. Phân Tích & Luận Bàn Kết Quả Cho Luận Văn

> **Kết luận Khoa học Then chốt**:
> 1. **Sự Đánh Đổi Utility Nghiêm Trọng**: DP-SGD làm giảm Test Accuracy đáng kể do việc cắt gradient và nhiễu cộng dồn làm suy giảm khả năng trích xuất đặc trưng của mạng Client.
> 2. **Sự Thất Bại của DP-SGD Trước Reconstruction Attack**: Dù mô hình đạt chuẩn riêng tư nghiêm ngặt ($\epsilon \approx 1.33$ tại $\sigma_{\mathrm{DP}}=2.0$), biểu diễn trung gian $z = F_c(x)$ vẫn bảo toàn thông tin cấu trúc của ảnh $x$. Decoder vẫn đạt PSNR $\approx 20$ dB và SSIM $> 0.65$, tái tạo rõ ràng hình dạng xe, tàu, máy bay, ngựa...
> 3. **Ý Nghĩa Thực Tiễn**: Chứng minh trực tiếp rằng cơ chế bảo mật tham số mô hình (Parameter DP) không thể bảo vệ dòng dữ liệu (Data Stream / Intermediate Representations) trong Split Learning, làm đòn bẩy vững chắc cho phương pháp **Task-Aware Perceptual Encryption (Bước 5)**.